# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll step through loading metadata, exploring structure, extracting records, performing basic processing and visualization, referencing all entities by their `@id`.

### Dataset Source
The dataset is described by a Croissant schema:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (using .metadata, not as a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are referenced by their `@id`. Let's enumerate the record sets, their fields, and columns.

In [ ]:
# List record sets available
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rset in record_sets:
    print(f"  @id: {rset['@id']}")

# Print fields and columns for each record set
for rset in record_sets:
    print(f"\nRecord Set @id: {rset['@id']}")
    fields = rset.get('fields', [])
    if fields:
        print("  Fields:")
        for fld in fields:
            print(f"    field @id: {fld['@id']}")
            columns = fld.get('columns', [])
            if columns:
                print("      Columns:")
                for col in columns:
                    print(f"        column @id: {col['@id']}, name: {col.get('name')}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All data is referenced by the corresponding `@id`.

Below, we'll show how to extract records from each record set, and display the columns.

In [ ]:
# List record set @ids for extraction
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

# Extract and load each record set as DataFrame
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded DataFrame from Record Set {rset_id}:")
        print("  Columns:", df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records from {rset_id}: {str(e)}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All references are made by `@id`.

We'll select a numeric field and a grouping field based on the previous overview. Adjust these variables as needed.

In [ ]:
# For demonstration, pick the largest record set
main_record_set_id = None
main_df = None
for rset_id, df in dataframes.items():
    if df.shape[0] > 0:
        main_record_set_id = rset_id
        main_df = df
        break

if main_df is not None:
    print(f"Using record set: {main_record_set_id}")
    # Choose numeric and group fields by their column names / @id
    # (Example: 'log_likelihood', 'ward') -- adjust if schema differs
    numeric_field_id = None
    group_field_id = None

    # Try to select appropriate columns
    numeric_candidates = [col for col in main_df.columns if 'log_likelihood' in col or 'coefficient' in col or 'value' in col or 'score' in col]
    group_candidates = [col for col in main_df.columns if 'ward' in col or 'county' in col or 'gender' in col]

    # Prioritize log_likelihood
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    if group_candidates:
        group_field_id = group_candidates[0]

    # Filtering: Remove NaNs and filter by numeric value
    threshold = 10
    filtered_df = main_df.dropna(subset=[numeric_field_id])
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head(3))

    # Grouping by group_field_id
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head(3))
else:
    print("No main DataFrame found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
We'll use matplotlib to show distributions for the selected numeric field.

In [ ]:
if main_df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    plt.hist(main_df[numeric_field_id].dropna(), bins=20, color='dodgerblue', alpha=0.7)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None:
        # Boxplot by group field
        plt.figure(figsize=(8, 6))
        main_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing the FAIR² dataset using Croissant metadata and the `mlcroissant` API. We referenced all entities by their `@id`, and provided an example workflow for extracting and analyzing records, normalizing numeric fields, and visualizing results.

Key observations may include data bias, missingness, and variable distributions, as outlined in the dataset metadata. Please refer to the Croissant schema and documentation for further study.

For more details, visit the [FAIR² dataset schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).